In [ ]:
import pandas as pd
import numpy as np
import re
import spacy

# ── SETUP ────────────────────────────────────────────────────────────────────
# Only NER needed — disable unused components for speed
nlp   = spacy.load("en_core_web_sm", disable=["tagger", "parser", "lemmatizer"])

# ── LOAD & PREPROCESS ────────────────────────────────────────────────────────
combined_data = pd.read_csv("../data/processed/combined_base_data.csv")
df = combined_data.copy()
df["datetime_posted"] = pd.to_datetime(df["datetime_posted"], utc=True, format="mixed")
df = df.dropna(subset=["title"]).copy()
df = df.reset_index(drop=True)

print(f"Loaded {len(df):,} rows | Fox%: {df['is_fox'].mean():.2%}")

# ── FEATURE EXTRACTION ───────────────────────────────────────────────────────
# EDA-justified features only. See stylometric_eda.py for selection rationale.
# Excluded features showed overlapping distributions between Fox and NBC.

def extract_features(headline: str) -> dict:
    text      = str(headline)
    words     = text.split()
    n_words   = max(len(words), 1)
    doc       = nlp(text)
    ent_types = [ent.label_ for ent in doc.ents]
    n_ents    = max(len(doc.ents), 1)

    return {
        "has_colon":           int(":" in text),                                     # Fox ~35% vs NBC ~13%
        "starts_with_number":  int(bool(re.match(r"^\d", text))),                   # NBC ~4x more than Fox
        "n_words":             n_words,                                              # Fox skews longer
        "char_count":          len(text),                                            # corroborates n_words
        "person_to_ent_ratio": ent_types.count("PERSON") / n_ents,                  # Fox leads with people
        "has_question":        int(text.endswith("?")),                              # Fox uses more questions
        "allcaps_word_count":  sum(1 for w in words if w.isupper() and len(w) > 1), # slight Fox lean
    }

print("Extracting features (spaCy NER — ~30s)...")
feature_df = df["title"].apply(extract_features).apply(pd.Series)

# Attach label and preserve original index
feature_df["is_fox"] = df["is_fox"].values
feature_df["datetime_posted"] = df["datetime_posted"].values
feature_df["title"] = df["title"].values

print("Done. Feature shape:", feature_df.shape)
print(feature_df.head())

Loaded 3,801 rows | Fox%: 52.62%
Extracting features (spaCy NER — ~30s)...
Done. Feature shape: (3801, 10)
   has_colon  starts_with_number  n_words  char_count  person_to_ent_ratio  \
0        0.0                 0.0     17.0       107.0             0.000000   
1        0.0                 0.0     12.0        82.0             1.000000   
2        0.0                 0.0     15.0        97.0             0.000000   
3        1.0                 0.0     22.0       107.0             1.000000   
4        0.0                 0.0     12.0        82.0             0.666667   

   has_question  allcaps_word_count  is_fox     datetime_posted  \
0           0.0                 1.0       1 2023-10-13 18:06:08   
1           0.0                 0.0       1 2024-10-18 19:56:05   
2           0.0                 1.0       1 2024-08-20 01:00:35   
3           0.0                 1.0       1 2023-06-09 17:55:28   
4           0.0                 0.0       1 2024-06-06 08:30:24   

                     

In [5]:
# ── SAVE ─────────────────────────────────────────────────────────────────────
out_path = "../data/processed/style_features.csv"
feature_df.to_csv(out_path, index=True)
print(f"\nSaved to {out_path}")


Saved to ../data/processed/style_features.csv
